# 🔬 Preprocessing Data Makroekonomi
## Analisis Deteksi Anomali Makroekonomi — Early Warning System Krisis Ekonomi

**Tahapan Preprocessing:**
1. Import Library & Load Data
2. Eksplorasi Awal (EDA Ringkas)
3. Penanganan Missing Value
4. Analisis Distribusi Data & Normalisasi
5. Deteksi dan Penanganan Outlier
6. Cek Keseimbangan Data (Labeling + SMOTE/Oversampling)
7. Korelasi Antar Fitur
8. Simpan Data Bersih

---

## 1. Import Library & Load Data

In [ ]:
# Install dependencies jika belum tersedia
# !pip install pandas numpy matplotlib seaborn scipy scikit-learn imbalanced-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.impute import KNNImputer
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print('✅ Library berhasil dimuat.')

In [ ]:
# Load dataset
df = pd.read_csv('data_before clean.csv')
print(f'Shape: {df.shape}')
print(f'Jumlah negara: {df["economy"].nunique()}')
print(f'Rentang tahun: {df["year"].min()} - {df["year"].max()}')
df.head(10)

## 2. Eksplorasi Awal (EDA Ringkas)

In [ ]:
# Info tipe data
df.info()

In [ ]:
# Statistik deskriptif
df.describe().T.round(3)

In [ ]:
# Daftar fitur numerik (exclude economy & year)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in numeric_cols if c != 'year']
print(f'Fitur numerik ({len(feature_cols)}): {feature_cols}')

In [ ]:
# Daftar negara dalam dataset
economies = sorted(df['economy'].unique())
print(f'\nDaftar {len(economies)} negara/ekonomi:')
print(economies)

---
## 3. Penanganan Missing Value

### 3.1 Identifikasi Missing Value

In [ ]:
# Hitung missing value per kolom
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing %', ascending=False)

print('='*50)
print('RINGKASAN MISSING VALUE')
print('='*50)
print(missing_df[missing_df['Missing Count'] > 0])
print(f'\nTotal sel missing: {df.isnull().sum().sum()}')
print(f'Total sel: {df.shape[0] * df.shape[1]}')
print(f'Persentase total: {(df.isnull().sum().sum() / (df.shape[0]*df.shape[1]) * 100):.2f}%')

In [ ]:
# Visualisasi missing value
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Bar chart missing
missing_filtered = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %')
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(missing_filtered)))
missing_filtered['Missing %'].plot(kind='barh', ax=axes[0], color=colors)
axes[0].set_title('Persentase Missing Value per Kolom', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Missing (%)')
for i, (idx, row) in enumerate(missing_filtered.iterrows()):
    axes[0].text(row['Missing %'] + 0.3, i, f"{row['Missing %']}%", va='center', fontsize=9)

# Heatmap missing per negara (top 10 missing columns)
top_missing_cols = missing_df[missing_df['Missing Count'] > 0].head(10).index.tolist()
if top_missing_cols:
    missing_by_country = df.groupby('economy')[top_missing_cols].apply(lambda x: x.isnull().sum())
    # Ambil 20 negara dengan missing terbanyak
    top_countries = missing_by_country.sum(axis=1).nlargest(20).index
    sns.heatmap(missing_by_country.loc[top_countries], cmap='YlOrRd', annot=True, fmt='g',
                linewidths=0.5, ax=axes[1], cbar_kws={'label': 'Jumlah Missing'})
    axes[1].set_title('Missing Value per Negara (Top 20 Negara)', fontsize=13, fontweight='bold')
    axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Visualisasi pattern missing value (msno-style)
fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(df[feature_cols].isnull().T, cbar=True, yticklabels=True,
            cmap=['#2ecc71', '#e74c3c'], ax=ax,
            cbar_kws={'label': '0=Ada, 1=Missing', 'ticks': [0, 1]})
ax.set_title('Pola Missing Value (Hijau=Ada, Merah=Missing)', fontsize=14, fontweight='bold')
ax.set_xlabel('Index Baris')
plt.tight_layout()
plt.show()

### 3.2 Strategi Penanganan Missing Value

Strategi yang digunakan:
1. **Kolom dengan missing > 50%**: Pertimbangkan drop atau imputasi khusus
2. **Data time-series per negara**: Gunakan interpolasi linear (forward/backward fill) per negara — karena data makroekonomi bersifat temporal
3. **Sisa missing**: Gunakan KNN Imputer untuk menangkap pola antar fitur
4. **Baris yang masih memiliki terlalu banyak missing** setelah imputasi: Drop

In [ ]:
# --- Step 1: Drop baris yang terlalu banyak missing (> 70% fitur kosong) ---
threshold = 0.7
n_features = len(feature_cols)
mask_too_many_missing = df[feature_cols].isnull().sum(axis=1) > (threshold * n_features)
print(f'Baris dengan > {threshold*100:.0f}% fitur missing: {mask_too_many_missing.sum()}')

df_clean = df[~mask_too_many_missing].copy()
print(f'Shape setelah drop baris terlalu banyak missing: {df_clean.shape}')

In [ ]:
# --- Step 2: Interpolasi per negara (time-series) ---
# Karena data makroekonomi bersifat sekuensial per negara per tahun,
# interpolasi linear adalah metode yang paling tepat.

df_clean = df_clean.sort_values(['economy', 'year']).reset_index(drop=True)

for col in feature_cols:
    df_clean[col] = df_clean.groupby('economy')[col].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both')
    )

missing_after_interp = df_clean[feature_cols].isnull().sum()
print('Missing setelah interpolasi per negara:')
print(missing_after_interp[missing_after_interp > 0])
print(f'\nTotal missing tersisa: {missing_after_interp.sum()}')

In [ ]:
# --- Step 3: KNN Imputer untuk sisa missing ---
remaining_missing = df_clean[feature_cols].isnull().sum().sum()

if remaining_missing > 0:
    print(f'Sisa missing: {remaining_missing} → Menggunakan KNN Imputer (k=5)...')
    imputer = KNNImputer(n_neighbors=5, weights='distance')
    df_clean[feature_cols] = imputer.fit_transform(df_clean[feature_cols])
    print(f'Missing setelah KNN Imputer: {df_clean[feature_cols].isnull().sum().sum()}')
else:
    print('✅ Tidak ada missing value tersisa setelah interpolasi.')

In [ ]:
# Verifikasi akhir missing value
print('='*50)
print('VERIFIKASI AKHIR MISSING VALUE')
print('='*50)
print(f'Total missing: {df_clean.isnull().sum().sum()}')
print(f'Shape data: {df_clean.shape}')
print(f'\n✅ Missing value telah ditangani sepenuhnya.' if df_clean.isnull().sum().sum() == 0 else '⚠️ Masih ada missing value.')

---
## 4. Analisis Distribusi Data & Normalisasi

### 4.1 Distribusi Sebelum Normalisasi

In [ ]:
# Histogram distribusi setiap fitur
n_cols = 4
n_rows = int(np.ceil(len(feature_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows*4))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes[i]
    data = df_clean[col].dropna()
    
    # Histogram + KDE
    ax.hist(data, bins=40, alpha=0.6, color='#3498db', edgecolor='white', density=True)
    data.plot.kde(ax=ax, color='#e74c3c', linewidth=2)
    
    # Statistik
    skewness = data.skew()
    kurtosis = data.kurtosis()
    ax.set_title(f'{col}\nSkew={skewness:.2f}, Kurt={kurtosis:.2f}', fontsize=10, fontweight='bold')
    ax.axvline(data.mean(), color='green', linestyle='--', alpha=0.7, label=f'Mean={data.mean():.2f}')
    ax.axvline(data.median(), color='orange', linestyle='--', alpha=0.7, label=f'Median={data.median():.2f}')
    ax.legend(fontsize=7)

# Kosongkan axes yang tidak terpakai
for j in range(len(feature_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribusi Fitur Sebelum Normalisasi', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Ringkasan skewness & kurtosis
skew_kurt = pd.DataFrame({
    'Skewness': df_clean[feature_cols].skew(),
    'Kurtosis': df_clean[feature_cols].kurtosis(),
    'Mean': df_clean[feature_cols].mean(),
    'Std': df_clean[feature_cols].std(),
    'Min': df_clean[feature_cols].min(),
    'Max': df_clean[feature_cols].max()
}).round(3)

# Interpretasi skewness
def interpret_skew(s):
    if abs(s) < 0.5: return '✅ Simetris'
    elif abs(s) < 1: return '⚠️ Agak Miring'
    else: return '❌ Sangat Miring'

skew_kurt['Interpretasi'] = skew_kurt['Skewness'].apply(interpret_skew)
print('Analisis Distribusi:')
skew_kurt

In [ ]:
# Shapiro-Wilk Test untuk normalitas (sampel max 5000)
print('='*60)
print('UJI NORMALITAS (Shapiro-Wilk Test, α=0.05)')
print('='*60)
normality_results = []
for col in feature_cols:
    data = df_clean[col].dropna()
    sample = data.sample(min(len(data), 5000), random_state=42)
    stat, p_value = stats.shapiro(sample)
    is_normal = 'Ya' if p_value > 0.05 else 'Tidak'
    normality_results.append({'Fitur': col, 'Statistic': round(stat, 4),
                               'P-Value': f'{p_value:.6f}', 'Normal?': is_normal})

normality_df = pd.DataFrame(normality_results)
print(normality_df.to_string(index=False))
print(f"\nFitur normal: {normality_df[normality_df['Normal?']=='Ya'].shape[0]}/{len(feature_cols)}")

### 4.2 Normalisasi/Standarisasi

**Strategi:**
- Karena data makroekonomi mengandung outlier dan distribusi skewed, kita menggunakan **RobustScaler** sebagai metode utama (robust terhadap outlier).
- Kita juga menyimpan versi **StandardScaler** dan **MinMaxScaler** untuk perbandingan.
- Untuk anomaly detection (Isolation Forest, LOF, Autoencoder), **StandardScaler** umumnya lebih cocok.

In [ ]:
# Terapkan beberapa scaler untuk perbandingan
scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

scaled_data = {}
for name, scaler in scalers.items():
    scaled_data[name] = pd.DataFrame(
        scaler.fit_transform(df_clean[feature_cols]),
        columns=feature_cols,
        index=df_clean.index
    )
    print(f'{name}: min={scaled_data[name].min().min():.3f}, max={scaled_data[name].max().max():.3f}')

In [ ]:
# Visualisasi perbandingan scaler
sample_features = feature_cols[:6]  # Ambil 6 fitur pertama untuk visualisasi

fig, axes = plt.subplots(3, len(sample_features), figsize=(24, 10))

for i, (name, data) in enumerate(scaled_data.items()):
    for j, col in enumerate(sample_features):
        ax = axes[i][j]
        ax.hist(data[col], bins=40, alpha=0.7, color=['#3498db', '#e74c3c', '#2ecc71'][i],
                edgecolor='white', density=True)
        if j == 0:
            ax.set_ylabel(name, fontsize=11, fontweight='bold')
        if i == 0:
            ax.set_title(col, fontsize=10, fontweight='bold')

plt.suptitle('Perbandingan Metode Normalisasi/Standarisasi', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Pilih StandardScaler sebagai metode utama untuk anomaly detection
# Alasan: model Isolation Forest, LOF, dan Autoencoder bekerja optimal dengan data terstandarkan

scaler_main = StandardScaler()
df_clean[feature_cols] = scaler_main.fit_transform(df_clean[feature_cols])

print('✅ Data telah dinormalisasi menggunakan StandardScaler')
print(f'Mean ~ 0: {df_clean[feature_cols].mean().mean():.6f}')
print(f'Std ~ 1: {df_clean[feature_cols].std().mean():.6f}')
df_clean[feature_cols].describe().T.round(3)

---
## 5. Deteksi dan Penanganan Outlier

### 5.1 Identifikasi Outlier

In [ ]:
# Box plot untuk identifikasi outlier
fig, axes = plt.subplots(2, 1, figsize=(18, 12))

# Sebelum penanganan
df_clean[feature_cols].plot(kind='box', ax=axes[0], vert=True, patch_artist=True,
                             boxprops=dict(facecolor='lightblue', alpha=0.7),
                             flierprops=dict(marker='o', markerfacecolor='red', markersize=3, alpha=0.5))
axes[0].set_title('Boxplot Semua Fitur (Setelah Standarisasi)', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].axhline(y=3, color='red', linestyle='--', alpha=0.5, label='±3σ')
axes[0].axhline(y=-3, color='red', linestyle='--', alpha=0.5)
axes[0].legend()

# Violin plot
parts = axes[1].violinplot([df_clean[col].dropna().values for col in feature_cols],
                           showmeans=True, showmedians=True)
axes[1].set_xticks(range(1, len(feature_cols)+1))
axes[1].set_xticklabels(feature_cols, rotation=45, ha='right')
axes[1].set_title('Violin Plot Distribusi Fitur', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Hitung jumlah outlier dengan IQR method & Z-score method
print('='*70)
print('DETEKSI OUTLIER: IQR Method vs Z-Score Method')
print('='*70)

outlier_summary = []

for col in feature_cols:
    data = df_clean[col].dropna()
    
    # IQR Method
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_iqr = ((data < lower) | (data > upper)).sum()
    
    # Z-Score Method (|z| > 3)
    z_scores = np.abs(stats.zscore(data))
    n_zscore = (z_scores > 3).sum()
    
    outlier_summary.append({
        'Fitur': col,
        'Outlier (IQR)': n_iqr,
        '% IQR': round(n_iqr/len(data)*100, 2),
        'Outlier (Z>3)': n_zscore,
        '% Z-Score': round(n_zscore/len(data)*100, 2)
    })

outlier_df = pd.DataFrame(outlier_summary)
print(outlier_df.to_string(index=False))
print(f'\nTotal outlier (IQR): {outlier_df["Outlier (IQR)"].sum()}')
print(f'Total outlier (Z-Score): {outlier_df["Outlier (Z>3)"].sum()}')

### 5.2 Penanganan Outlier

**Strategi:** Untuk deteksi anomali, **outlier TIDAK dihapus**, karena outlier justru bisa menjadi **indikator krisis/anomali** yang ingin dideteksi.

Namun, kita tetap melakukan **Winsorizing (capping)** pada nilai yang sangat ekstrem (> 5σ) untuk mencegah overfitting pada noise, bukan sinyal krisis.

**Alasan:**
- Inflasi Argentina (`Inflation_CPI` > 200%) atau crash GDP -10% adalah sinyal krisis yang valid
- Nilai > 5σ umumnya merupakan error data atau kejadian sangat langka

In [ ]:
# Winsorizing: Cap nilai di luar 5-sigma (persentil 0.5% dan 99.5%)
df_capped = df_clean.copy()
cap_count = 0

for col in feature_cols:
    lower_bound = df_capped[col].quantile(0.005)
    upper_bound = df_capped[col].quantile(0.995)
    
    n_capped = ((df_capped[col] < lower_bound) | (df_capped[col] > upper_bound)).sum()
    cap_count += n_capped
    
    df_capped[col] = df_capped[col].clip(lower=lower_bound, upper=upper_bound)

print(f'Total nilai yang di-cap (winsorized): {cap_count}')
print(f'Persentase: {cap_count / (len(df_capped) * len(feature_cols)) * 100:.2f}%')

In [ ]:
# Visualisasi sebelum vs sesudah capping
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Sebelum
df_clean[feature_cols].plot(kind='box', ax=axes[0], vert=True, patch_artist=True,
                            boxprops=dict(facecolor='#ffcccc', alpha=0.7),
                            flierprops=dict(marker='o', markerfacecolor='red', markersize=3, alpha=0.5))
axes[0].set_title('SEBELUM Winsorizing', fontsize=13, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Sesudah
df_capped[feature_cols].plot(kind='box', ax=axes[1], vert=True, patch_artist=True,
                             boxprops=dict(facecolor='#ccffcc', alpha=0.7),
                             flierprops=dict(marker='o', markerfacecolor='green', markersize=3, alpha=0.5))
axes[1].set_title('SESUDAH Winsorizing (Cap 0.5%-99.5%)', fontsize=13, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Perbandingan Outlier Sebelum & Sesudah Winsorizing', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Update dataframe utama
df_clean = df_capped.copy()

---
## 6. Cek Keseimbangan Data (Labeling + Resampling)

### 6.1 Membuat Label Anomali (Crisis Label)

Untuk deteksi anomali, kita perlu membuat label krisis berdasarkan indikator makroekonomi.

**Definisi Krisis Ekonomi (Multi-Indikator):**
- GDP Growth < -2% (kontraksi ekonomi signifikan)
- ATAU Inflation CPI > 2σ dari mean (hiperinflasi relatif)
- ATAU Unemployment naik > 1.5σ dari mean negara tersebut
- ATAU Current Account deficit > 2σ

> **Catatan:** Label ini bersifat proxy/heuristik. Untuk riset lebih lanjut, gunakan data krisis historis (Laeven & Valencia, IMF).

In [ ]:
# Labeling berbasis rule — menggunakan data sebelum standarisasi
# Kita perlu reload data asli yang sudah dibersihkan missing value tapi belum di-scale

# Re-load dan re-process tanpa scaling untuk labeling
df_label = pd.read_csv('data_before clean.csv')
df_label = df_label.sort_values(['economy', 'year']).reset_index(drop=True)

# Hapus baris terlalu banyak missing
mask_too_many = df_label[feature_cols].isnull().sum(axis=1) > (0.7 * len(feature_cols))
df_label = df_label[~mask_too_many].copy()

# Interpolasi
for col in feature_cols:
    df_label[col] = df_label.groupby('economy')[col].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both')
    )

# KNN Imputer
remaining = df_label[feature_cols].isnull().sum().sum()
if remaining > 0:
    imp = KNNImputer(n_neighbors=5, weights='distance')
    df_label[feature_cols] = imp.fit_transform(df_label[feature_cols])

print(f'Data untuk labeling: {df_label.shape}')
print(f'Missing: {df_label[feature_cols].isnull().sum().sum()}')

In [ ]:
# Buat label krisis
def create_crisis_label(df):
    """
    Membuat label krisis berdasarkan beberapa indikator makroekonomi.
    1 = Krisis/Anomali, 0 = Normal
    """
    labels = pd.Series(0, index=df.index)
    
    # Kriteria 1: Kontraksi GDP signifikan (GDP_Growth < -2%)
    if 'GDP_Growth' in df.columns:
        crit_gdp = df['GDP_Growth'] < -2
        labels = labels | crit_gdp.astype(int)
    
    # Kriteria 2: Inflasi sangat tinggi (> mean + 2*std per negara)
    if 'Inflation_CPI' in df.columns:
        inflation_threshold = df.groupby('economy')['Inflation_CPI'].transform(
            lambda x: x.mean() + 2 * x.std()
        )
        crit_inflation = df['Inflation_CPI'] > inflation_threshold
        labels = labels | crit_inflation.astype(int)
    
    # Kriteria 3: Unemployment sangat tinggi (> mean + 1.5*std per negara)
    if 'Unemployment' in df.columns:
        unemp_threshold = df.groupby('economy')['Unemployment'].transform(
            lambda x: x.mean() + 1.5 * x.std()
        )
        crit_unemp = df['Unemployment'] > unemp_threshold
        labels = labels | crit_unemp.astype(int)
    
    # Kriteria 4: Current Account deficit sangat besar (> mean - 2*std)
    if 'Current_Account_GDP' in df.columns:
        ca_threshold = df.groupby('economy')['Current_Account_GDP'].transform(
            lambda x: x.mean() - 2 * x.std()
        )
        crit_ca = df['Current_Account_GDP'] < ca_threshold
        labels = labels | crit_ca.astype(int)
    
    return labels

df_label['crisis_label'] = create_crisis_label(df_label)

# Juga tambahkan ke df_clean
df_clean = df_clean.reset_index(drop=True)
df_label = df_label.reset_index(drop=True)

# Pastikan index cocok
df_clean['crisis_label'] = df_label['crisis_label'].values

print('\n--- Distribusi Label Krisis ---')
print(df_label['crisis_label'].value_counts())
print(f'\nPersentase krisis: {df_label["crisis_label"].mean()*100:.2f}%')

In [ ]:
# Visualisasi distribusi label
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Pie chart
counts = df_clean['crisis_label'].value_counts()
colors = ['#2ecc71', '#e74c3c']
labels_pie = ['Normal (0)', 'Krisis (1)']
explode = [0, 0.1]
axes[0].pie(counts, explode=explode, labels=labels_pie, colors=colors,
            autopct='%1.1f%%', shadow=True, startangle=90, textprops={'fontsize': 12})
axes[0].set_title('Distribusi Label', fontsize=13, fontweight='bold')

# Bar chart
counts.plot(kind='bar', ax=axes[1], color=colors, edgecolor='white', alpha=0.8)
axes[1].set_title('Jumlah Sampel per Label', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Label')
axes[1].set_ylabel('Jumlah')
axes[1].set_xticklabels(['Normal', 'Krisis'], rotation=0)
for i, v in enumerate(counts):
    axes[1].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Distribusi krisis per tahun
crisis_by_year = df_clean.groupby(df_label['year'])['crisis_label'].mean() * 100
crisis_by_year.plot(kind='bar', ax=axes[2], color='#e74c3c', alpha=0.7, edgecolor='white')
axes[2].set_title('Persentase Krisis per Tahun', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Tahun')
axes[2].set_ylabel('% Krisis')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 6.2 Penanganan Ketidakseimbangan Data

Karena anomali/krisis umumnya **jarang terjadi** (imbalanced), kita menggunakan beberapa teknik:
1. **SMOTE** (Synthetic Minority Over-sampling Technique)
2. **ADASYN** (Adaptive Synthetic Sampling)

Kita bandingkan keduanya dan pilih yang terbaik.

In [ ]:
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from collections import Counter

X = df_clean[feature_cols].values
y = df_clean['crisis_label'].values

print(f'Distribusi SEBELUM resampling: {Counter(y)}')
print(f'Rasio: 1:{Counter(y)[0]//max(Counter(y)[1],1)}')

In [ ]:
# --- SMOTE ---
smote = SMOTE(random_state=42, k_neighbors=5)
X_smote, y_smote = smote.fit_resample(X, y)
print(f'SMOTE: {Counter(y_smote)}')

# --- ADASYN ---
try:
    adasyn = ADASYN(random_state=42, n_neighbors=5)
    X_adasyn, y_adasyn = adasyn.fit_resample(X, y)
    print(f'ADASYN: {Counter(y_adasyn)}')
except Exception as e:
    print(f'ADASYN gagal (kemungkinan data sudah cukup seimbang): {e}')
    X_adasyn, y_adasyn = X_smote.copy(), y_smote.copy()

# --- SMOTE + ENN (Kombinasi) ---
try:
    smoteenn = SMOTEENN(random_state=42)
    X_smoteenn, y_smoteenn = smoteenn.fit_resample(X, y)
    print(f'SMOTE+ENN: {Counter(y_smoteenn)}')
except Exception as e:
    print(f'SMOTE+ENN gagal: {e}')
    X_smoteenn, y_smoteenn = X_smote.copy(), y_smote.copy()

In [ ]:
# Visualisasi perbandingan
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

datasets = [
    ('Original', Counter(y)),
    ('SMOTE', Counter(y_smote)),
    ('ADASYN', Counter(y_adasyn)),
    ('SMOTE+ENN', Counter(y_smoteenn))
]

colors = ['#2ecc71', '#e74c3c']

for i, (name, counter) in enumerate(datasets):
    values = [counter[0], counter[1]]
    bars = axes[i].bar(['Normal', 'Krisis'], values, color=colors, edgecolor='white', alpha=0.8)
    axes[i].set_title(f'{name}\n(Total: {sum(values)})', fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Jumlah')
    for bar, v in zip(bars, values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                     str(v), ha='center', fontweight='bold')

plt.suptitle('Perbandingan Teknik Resampling', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Pilih SMOTE sebagai metode resampling utama
# Alasan: paling stabil dan menghasilkan distribusi seimbang sempurna

df_resampled = pd.DataFrame(X_smote, columns=feature_cols)
df_resampled['crisis_label'] = y_smote

print(f'\n✅ Dataset setelah SMOTE:')
print(f'Shape: {df_resampled.shape}')
print(f'Distribusi: {Counter(y_smote)}')
print(f'Rasio: 1:1 (Seimbang)')

---
## 7. Analisis Korelasi

In [ ]:
# Correlation Matrix
fig, axes = plt.subplots(1, 2, figsize=(22, 8))

# Pearson correlation
corr_matrix = df_clean[feature_cols].corr(method='pearson')
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=axes[0],
            cbar_kws={'label': 'Korelasi Pearson'},
            annot_kws={'fontsize': 8})
axes[0].set_title('Matriks Korelasi — Pearson', fontsize=13, fontweight='bold')

# Spearman correlation (lebih robust terhadap non-linearitas)
corr_spearman = df_clean[feature_cols].corr(method='spearman')
sns.heatmap(corr_spearman, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=axes[1],
            cbar_kws={'label': 'Korelasi Spearman'},
            annot_kws={'fontsize': 8})
axes[1].set_title('Matriks Korelasi — Spearman', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Identifikasi fitur yang berkorelasi tinggi (> 0.8)
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.8:
            high_corr_pairs.append({
                'Fitur 1': corr_matrix.columns[i],
                'Fitur 2': corr_matrix.columns[j],
                'Korelasi': round(corr_matrix.iloc[i, j], 3)
            })

if high_corr_pairs:
    print('='*50)
    print('FITUR DENGAN KORELASI TINGGI (|r| > 0.8)')
    print('='*50)
    high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('Korelasi', key=abs, ascending=False)
    print(high_corr_df.to_string(index=False))
    print(f'\n⚠️ {len(high_corr_pairs)} pasangan fitur memiliki korelasi tinggi.')
    print('Pertimbangkan untuk menghapus salah satu fitur atau menggunakan PCA.')
else:
    print('✅ Tidak ada pasangan fitur dengan korelasi > 0.8')

---
## 8. Ringkasan & Simpan Data Bersih

In [ ]:
# Ringkasan preprocessing
print('='*60)
print('📊 RINGKASAN PREPROCESSING')
print('='*60)
print(f'\n1. DATA AWAL')
print(f'   Shape: {df.shape}')
print(f'   Missing values: {df.isnull().sum().sum()}')

print(f'\n2. DATA SETELAH PEMBERSIHAN (tanpa SMOTE)')
print(f'   Shape: {df_clean.shape}')
print(f'   Missing values: {df_clean[feature_cols].isnull().sum().sum()}')
print(f'   Metode missing value: Interpolasi Linear + KNN Imputer')
print(f'   Normalisasi: StandardScaler')
print(f'   Outlier: Winsorizing (cap 0.5%-99.5%)')

print(f'\n3. LABEL ANOMALI')
crisis_count = Counter(df_clean['crisis_label'].values)
print(f'   Normal: {crisis_count[0]} ({crisis_count[0]/len(df_clean)*100:.1f}%)')
print(f'   Krisis: {crisis_count[1]} ({crisis_count[1]/len(df_clean)*100:.1f}%)')

print(f'\n4. DATA SETELAH SMOTE')
print(f'   Shape: {df_resampled.shape}')
smote_count = Counter(df_resampled['crisis_label'].values)
print(f'   Normal: {smote_count[0]} ({smote_count[0]/len(df_resampled)*100:.1f}%)')
print(f'   Krisis: {smote_count[1]} ({smote_count[1]/len(df_resampled)*100:.1f}%)')

print(f'\n5. FITUR ({len(feature_cols)} fitur):')
for i, col in enumerate(feature_cols, 1):
    print(f'   {i:2d}. {col}')

print('\n' + '='*60)

In [ ]:
# Simpan dataset bersih

# 1. Data tanpa SMOTE (untuk unsupervised anomaly detection)
df_save = df_clean.copy()
df_save.to_csv('data_cleaned.csv', index=False)
print(f'✅ data_cleaned.csv disimpan ({df_save.shape})')

# 2. Data dengan SMOTE (untuk supervised / semi-supervised)
df_resampled.to_csv('data_cleaned_smote.csv', index=False)
print(f'✅ data_cleaned_smote.csv disimpan ({df_resampled.shape})')

# 3. Data label saja (tanpa scaling, untuk referensi)
df_label_save = df_label[['economy', 'year'] + feature_cols + ['crisis_label']].copy()
df_label_save.to_csv('data_with_labels.csv', index=False)
print(f'✅ data_with_labels.csv disimpan ({df_label_save.shape})')

In [ ]:
# Preview data final
print('\n📋 Preview Data Bersih (tanpa SMOTE):')
df_clean.head(10)

In [ ]:
# Preview data SMOTE
print('\n📋 Preview Data SMOTE:')
df_resampled.head(10)

---
## ✅ Kesimpulan Preprocessing

| Tahap | Metode | Alasan |
|-------|--------|--------|
| **Missing Value** | Interpolasi Linear + KNN Imputer | Data time-series → interpolasi temporal; KNN untuk pola antar fitur |
| **Normalisasi** | StandardScaler | Cocok untuk Isolation Forest, LOF, Autoencoder (mean=0, std=1) |
| **Outlier** | Winsorizing (0.5%-99.5%) | Pertahankan outlier krisis, hanya cap nilai sangat ekstrem |
| **Keseimbangan** | SMOTE | Mengatasi imbalanced class untuk evaluasi supervised |
| **Korelasi** | Pearson + Spearman | Identifikasi multikolinearitas |

**Output file:**
- `data_cleaned.csv` → Data bersih tanpa SMOTE (untuk unsupervised)
- `data_cleaned_smote.csv` → Data bersih dengan SMOTE (untuk supervised)
- `data_with_labels.csv` → Data dengan label tanpa scaling (referensi)